<a href="https://colab.research.google.com/github/kingfaluk/KCCA_FC/blob/main/week3_sessiosn_homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    OrdinalEncoder,
    OneHotEncoder
)
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [3]:
df = pd.read_csv("water_potability.csv")

df.head()

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0


In [4]:
numeric_features = [
    "ph",
    "Hardness",
    "Solids",
    "Chloramines",
    "Sulfate",
    "Conductivity",
    "Organic_carbon",
    "Trihalomethanes",
    "Turbidity"
]

hardness_labels = [
    "Soft",
    "Moderate",
    "Hard",
    "Very Hard"
]

In [5]:
data = df.copy()

data["hardness_level"] = pd.cut(
    data["Hardness"],
    bins=[0, 60, 120, 180, float("inf")],
    labels=hardness_labels
)

data["carbon_thm_interaction"] = (
    data["Organic_carbon"] * data["Trihalomethanes"]
)

data["solids_per_conductivity"] = (
    data["Solids"] / data["Conductivity"]
)

data["chloramine_turbidity_interaction"] = (
    data["Chloramines"] * data["Turbidity"]
)

data.head()

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability,hardness_level,carbon_thm_interaction,solids_per_conductivity,chloramine_turbidity_interaction
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0,Very Hard,902.947403,36.843878,21.631516
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0,Hard,855.076117,31.422698,29.862961
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0,Very Hard,1120.416425,47.561506,28.346486
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0,Very Hard,1849.951737,60.612296,37.304800
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0,Very Hard,369.841742,45.126753,26.677889


In [6]:
all_numeric = numeric_features + [
    "carbon_thm_interaction",
    "solids_per_conductivity",
    "chloramine_turbidity_interaction"
]

X = data[all_numeric + ["hardness_level"]]
y = data["Potability"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [7]:
def build_pipeline(num_cols, scaler):

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", scaler)
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(categories=[hardness_labels]))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, num_cols),
        ("categorical", categorical_pipeline, ["hardness_level"])
    ])

    model = Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ])

    return model

In [ ]:
model_new = build_pipeline(
    all_numeric,
    StandardScaler()
)

model_new.fit(X_train, y_train)

predictions = model_new.predict(X_test)

acc_new = accuracy_score(
    y_test,
    predictions
)

print("Accuracy with new feature:", round(acc_new, 3))

In [ ]:
baseline_numeric = numeric_features + [
    "carbon_thm_interaction",
    "solids_per_conductivity"
]

model_base = build_pipeline(
    baseline_numeric,
    StandardScaler()
)

model_base.fit(
    X_train[baseline_numeric + ["hardness_level"]],
    y_train
)

predictions_base = model_base.predict(
    X_test[baseline_numeric + ["hardness_level"]]
)

acc_base = accuracy_score(
    y_test,
    predictions_base
)

print("With new feature   :", round(acc_new, 3))
print("Without new feature:", round(acc_base, 3))

In [ ]:
part8 = df.copy()

part8["ph_category"] = pd.cut(
    part8["ph"],
    bins=[-float("inf"), 6.5, 8.5, float("inf")],
    labels=["Acidic", "Neutral", "Alkaline"]
)

X = part8[numeric_features + ["ph_category"]]
y = part8["Potability"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
def scaler_pipeline(scaler):

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", scaler)
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, ["ph_category"])
    ])

    return Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ])

In [ ]:
standard_model = scaler_pipeline(
    StandardScaler()
)

standard_model.fit(
    X_train,
    y_train
)

standard_predictions = standard_model.predict(
    X_test
)

acc_standard = accuracy_score(
    y_test,
    standard_predictions
)

print("StandardScaler accuracy:", round(acc_standard, 3))

In [ ]:
minmax_model = scaler_pipeline(
    MinMaxScaler()
)

minmax_model.fit(
    X_train,
    y_train
)

minmax_predictions = minmax_model.predict(
    X_test
)

acc_minmax = accuracy_score(
    y_test,
    minmax_predictions
)

print("MinMaxScaler accuracy:", round(acc_minmax, 3))

In [ ]:
print("StandardScaler:", round(acc_standard, 3))
print("MinMaxScaler  :", round(acc_minmax, 3))

difference = abs(acc_standard - acc_minmax)

print("Difference    :", round(difference, 3))

In [ ]:
selection_data = df.copy()

for column in [
    "ph",
    "Sulfate",
    "Trihalomethanes"
]:
    selection_data[column] = selection_data[column].fillna(
        selection_data[column].median()
    )

X_selection = selection_data[numeric_features]
y_selection = selection_data["Potability"]

X_sel_train, X_sel_test, y_sel_train, y_sel_test = train_test_split(
    X_selection,
    y_selection,
    test_size=0.2,
    random_state=42
)

In [ ]:
selector5 = SelectKBest(
    score_func=f_classif,
    k=5
)

selector5.fit(
    X_sel_train,
    y_sel_train
)

top5 = list(
    X_sel_train.columns[
        selector5.get_support()
    ]
)

print("Top 5 features:")
print(top5)

In [ ]:
selector3 = SelectKBest(
    score_func=f_classif,
    k=3
)

selector3.fit(
    X_sel_train,
    y_sel_train
)

top3 = list(
    X_sel_train.columns[
        selector3.get_support()
    ]
)

print("Top 3 features:")
print(top3)

In [ ]:
def feature_pipeline(features):

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(categories=[hardness_labels]))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, features),
        ("categorical", categorical_pipeline, ["hardness_level"])
    ])

    return Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ])

In [ ]:
train_data = data.loc[X_sel_train.index]
test_data = data.loc[X_sel_test.index]

y_train = train_data["Potability"]
y_test = test_data["Potability"]

In [ ]:
full_model = feature_pipeline(
    baseline_numeric
)

full_model.fit(
    train_data[baseline_numeric + ["hardness_level"]],
    y_train
)

full_predictions = full_model.predict(
    test_data[baseline_numeric + ["hardness_level"]]
)

acc_full = accuracy_score(
    y_test,
    full_predictions
)

print("Full model accuracy:", round(acc_full, 3))

In [ ]:
k3_model = feature_pipeline(top3)

k3_model.fit(
    train_data[top3 + ["hardness_level"]],
    y_train
)

k3_predictions = k3_model.predict(
    test_data[top3 + ["hardness_level"]]
)

acc_k3 = accuracy_score(
    y_test,
    k3_predictions
)

print("Top 3 accuracy:", round(acc_k3, 3))

In [ ]:
print("Task 4 Results")
print("-------------------------")
print("Full model :", round(acc_full, 3))
print("Top 5      :", round(acc_k5, 3))
print("Top 3      :", round(acc_k3, 3))